# 03: Ornstein-Uhlenbeck 過程による気温モデリング

## なぜ OU過程か？ GBM ではダメなのか？

株価のモデリングでは **幾何ブラウン運動 (GBM)** が標準:

$$dS = \mu S \, dt + \sigma S \, dW$$

しかし、気温にGBMは不適切:

| 特性 | GBM (株価) | OU過程 (気温) |
|------|-----------|--------------|
| 長期的挙動 | 指数的に発散しうる | 季節的平均に回帰 |
| 分布 | 対数正規分布 | 正規分布（定常） |
| 平均回帰 | なし | あり（κで制御） |
| 定常分布 | 存在しない | σ²/(2κ) の分散を持つ正規分布 |
| 物理的意味 | 資産価値の成長 | 大気の熱慣性による平衡回帰 |

**気温は「平均に戻る」**。1月に異常に暖かい日があっても、数日で冬の平均気温に戻る。
この性質を捉えるのが **Ornstein-Uhlenbeck 過程**:

$$dT = \kappa(\theta(t) - T) \, dt + \sigma \, dW$$

- $\kappa$: 平均回帰速度（大きいほど速く戻る）
- $\theta(t)$: 時変の平均気温（季節成分）
- $\sigma$: ボラティリティ

In [ ]:
import sys
sys.path.insert(0, "../backend")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from datetime import date, timedelta

from app.services.noaa_client import generate_sample_data
from app.models.temperature import (
    fit_seasonal,
    fit_ou,
    fit_temperature_model,
    simulate_temperature_paths,
    SeasonalParams,
    OUParams,
)

plt.rcParams['figure.figsize'] = (14, 5)
plt.rcParams['figure.dpi'] = 100
sns.set_style("whitegrid")
print("Setup complete!")

In [ ]:
# 5年分のサンプルデータ（パラメータ推定の精度を上げるため長めに）
sample_data = generate_sample_data(
    start_date=date(2019, 1, 1),
    end_date=date(2023, 12, 31),
)

t = np.arange(len(sample_data), dtype=np.float64)
tavg = np.array([d.tavg for d in sample_data])
dates = [d.date for d in sample_data]

print(f"データ期間: {dates[0]} ~ {dates[-1]}")
print(f"データ点数: {len(tavg)}")

## 1. 季節成分 θ(t) のフィット

気温から「季節パターン」を分離する。モデル:

$$\theta(t) = a + bt + c_1 \sin\left(\frac{2\pi t}{365}\right) + d_1 \cos\left(\frac{2\pi t}{365}\right) + c_2 \sin\left(\frac{4\pi t}{365}\right) + d_2 \cos\left(\frac{4\pi t}{365}\right)$$

- 第1高調波（年周期）: 夏/冬の基本パターン
- 第2高調波（半年周期）: 春秋の非対称性

In [ ]:
# モデル全体をフィット
result = fit_temperature_model(t, tavg)

# 季節成分のパラメータ表示
s = result.seasonal
amplitude = np.sqrt(s.c1**2 + s.d1**2)

print("=== 季節成分パラメータ ===")
print(f"  年平均気温 (a): {s.a:.2f}°C")
print(f"  線形トレンド (b): {s.b:.6f}°C/日 ({s.b*365:.3f}°C/年)")
print(f"  年周期 振幅: {amplitude:.2f}°C")
print(f"  R² (季節成分の説明力): {result.r_squared:.4f}")
print(f"\n→ 季節成分だけで気温変動の {result.r_squared*100:.1f}% を説明できる")

In [ ]:
# 実データ vs 季節成分フィットの比較プロット
theta = result.seasonal.evaluate(t)
dates_dt = [pd.Timestamp(d) for d in dates]

fig, ax = plt.subplots(figsize=(16, 6))
ax.plot(dates_dt, tavg, color="steelblue", alpha=0.4, linewidth=0.5, label="Observed T_avg")
ax.plot(dates_dt, theta, color="red", linewidth=2, label="Seasonal component θ(t)")
ax.set_xlabel("Date")
ax.set_ylabel("Temperature (°C)")
ax.set_title("Observed Temperature vs Fitted Seasonal Component")
ax.legend()
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=6))
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 2. 残差分析 — OU過程のインプット

季節成分を除去した**残差** = T_observed − θ(t) が OU過程に従うと仮定する。

良いモデルなら残差は:
- 平均 ≈ 0
- 自己相関がある（今日暖かいなら明日も暖かい → 平均回帰の証拠）
- 正規分布に近い

In [ ]:
residuals = tavg - theta

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. 残差の時系列
axes[0].plot(dates_dt, residuals, color="navy", linewidth=0.5, alpha=0.7)
axes[0].axhline(0, color="red", linestyle="--", alpha=0.5)
axes[0].set_title("Residuals (Deseasonalized Temperature)")
axes[0].set_xlabel("Date")
axes[0].set_ylabel("Residual (°C)")

# 2. 残差のヒストグラム
axes[1].hist(residuals, bins=50, density=True, color="steelblue", alpha=0.7, edgecolor="white")
# 正規分布をオーバーレイ
x_norm = np.linspace(residuals.min(), residuals.max(), 100)
from scipy.stats import norm
axes[1].plot(x_norm, norm.pdf(x_norm, residuals.mean(), residuals.std()),
             color="red", linewidth=2, label=f"N({residuals.mean():.1f}, {residuals.std():.1f}²)")
axes[1].set_title("Residual Distribution")
axes[1].set_xlabel("Residual (°C)")
axes[1].legend()

# 3. 自己相関 (lag-1 scatter plot)
axes[2].scatter(residuals[:-1], residuals[1:], alpha=0.2, s=5, color="navy")
axes[2].set_xlabel("X(t)")
axes[2].set_ylabel("X(t+1)")
axes[2].set_title("Lag-1 Scatter Plot (AR(1) structure)")
# 回帰直線
phi = np.polyfit(residuals[:-1], residuals[1:], 1)[0]
x_line = np.array([residuals.min(), residuals.max()])
axes[2].plot(x_line, phi * x_line, color="red", linewidth=2, label=f"slope φ = {phi:.3f}")
axes[2].legend()

plt.tight_layout()
plt.show()

print(f"残差の統計量: mean={residuals.mean():.3f}, std={residuals.std():.3f}")
print(f"AR(1)係数 φ = {phi:.4f}")
print(f"→ φ < 1 なので平均回帰している（OU過程の証拠）")

## 3. OU過程のパラメータ推定

残差に対して OU過程 $dX = -\kappa X \, dt + \sigma \, dW$ のパラメータを推定する。

### パラメータの直感的な意味

- **κ (kappa)**: 平均回帰速度。「偏差が半分に戻るまで何日かかるか」= $\ln 2 / \kappa$
  - κ = 0.3 → 半減期 ≈ 2.3日（異常気温は2-3日で解消）
  - κ = 0.1 → 半減期 ≈ 6.9日（異常気温が1週間続く）
  
- **σ (sigma)**: ボラティリティ。日々のランダムショックの大きさ (°C/√日)

- **定常分散** = $\sigma^2 / (2\kappa)$: 長期的な気温のばらつき
  - κが大きい（速く戻る）& σが小さい（ショックが小さい）→ ばらつきが小さい

In [ ]:
ou = result.ou

print("=== OU過程パラメータ推定結果 ===")
print(f"  κ (mean-reversion speed): {ou.kappa:.4f} ± {ou.kappa_se:.4f} /day")
print(f"  σ (volatility):           {ou.sigma:.4f} ± {ou.sigma_se:.4f} °C/√day")
print(f"")
print(f"=== 導出量 ===")
print(f"  半減期:     {ou.half_life:.1f} 日")
print(f"  定常標準偏差: {ou.stationary_std:.2f} °C")
print(f"  定常分散:    {ou.stationary_variance:.2f} °C²")
print(f"")
print(f"=== 直感的解釈 ===")
print(f"  「気温偏差は約{ou.half_life:.0f}日で半分に戻り、")
print(f"   長期的な気温のばらつきは ±{ou.stationary_std:.1f}°C 程度」")

## 4. モンテカルロ シミュレーション

推定パラメータを使って、Euler-Maruyama法で将来の気温パスをシミュレートする。

$$T_{n+1} = T_n + \kappa(\theta(t_n) - T_n)\Delta t + \sigma\sqrt{\Delta t} \, Z_n, \quad Z_n \sim N(0,1)$$

これがStep 3のプライシングの基盤になる。

In [ ]:
# 1年先までの気温パスを10,000本シミュレート
n_days_sim = 365
n_paths = 10000

paths = simulate_temperature_paths(
    seasonal=result.seasonal,
    ou=result.ou,
    t_start=float(t[-1]),
    n_days=n_days_sim,
    n_paths=n_paths,
    T_initial=float(tavg[-1]),
    seed=42,
)

print(f"シミュレーション: {n_paths} paths × {n_days_sim} days")
print(f"Shape: {paths.shape}")

### Fan Chart — シミュレーション結果の可視化

Fan Chartはパーセンタイルバンドで将来の不確実性を表現する。
バンドが広いほど不確実性が大きい。

- **中心線 (p50)**: 予測の中央値
- **濃い帯 (p25-p75)**: 50%信頼区間
- **薄い帯 (p5-p95)**: 90%信頼区間

In [ ]:
# パーセンタイルを計算
p5  = np.percentile(paths, 5, axis=0)
p25 = np.percentile(paths, 25, axis=0)
p50 = np.percentile(paths, 50, axis=0)
p75 = np.percentile(paths, 75, axis=0)
p95 = np.percentile(paths, 95, axis=0)

# シミュレーション期間の日付
sim_start = dates[-1] + timedelta(days=1)
sim_dates = [sim_start + timedelta(days=i) for i in range(n_days_sim)]
sim_dates_dt = [pd.Timestamp(d) for d in sim_dates]

# 季節成分
t_sim = np.arange(float(t[-1]), float(t[-1]) + n_days_sim)
theta_sim = result.seasonal.evaluate(t_sim)

# プロット
fig, ax = plt.subplots(figsize=(16, 7))

# 過去データ（最後の1年分）
last_year_idx = max(0, len(dates) - 365)
hist_dates = [pd.Timestamp(d) for d in dates[last_year_idx:]]
ax.plot(hist_dates, tavg[last_year_idx:], color="navy", linewidth=0.8, alpha=0.7, label="Observed")

# Fan Chart
ax.fill_between(sim_dates_dt, p5, p95, alpha=0.15, color="steelblue", label="90% CI (p5-p95)")
ax.fill_between(sim_dates_dt, p25, p75, alpha=0.3, color="steelblue", label="50% CI (p25-p75)")
ax.plot(sim_dates_dt, p50, color="steelblue", linewidth=2, label="Median (p50)")

# 季節成分
ax.plot(sim_dates_dt, theta_sim, color="red", linewidth=1.5, linestyle="--", alpha=0.7, label="Seasonal θ(t)")

# いくつかのサンプルパスを薄く表示
for i in range(5):
    ax.plot(sim_dates_dt, paths[i], alpha=0.15, linewidth=0.5, color="gray")

ax.axvline(pd.Timestamp(dates[-1]), color="black", linestyle=":", alpha=0.5, label="Forecast start")
ax.set_xlabel("Date")
ax.set_ylabel("Temperature (°C)")
ax.set_title(f"Temperature Fan Chart — {n_paths} Monte Carlo Paths")
ax.legend(loc="upper right")
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=2))
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 5. GBM vs OU — パスの比較

GBM（株価モデル）で気温パスを生成するとどうなるか？
OU過程との違いを視覚的に確認する。

In [ ]:
rng = np.random.default_rng(42)
n_compare = 365
n_show = 5

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# OU過程パス（既にシミュレーション済み）
for i in range(n_show):
    axes[0].plot(range(n_compare), paths[i, :n_compare], alpha=0.6, linewidth=0.8)
axes[0].plot(range(n_compare), theta_sim[:n_compare], color="red", linewidth=2, linestyle="--", label="θ(t)")
axes[0].set_title("OU Process (Mean-Reverting)")
axes[0].set_xlabel("Days")
axes[0].set_ylabel("Temperature (°C)")
axes[0].legend()
axes[0].set_ylim(-20, 40)

# GBM パス（気温に不適切であることのデモ）
mu_gbm = 0.0  # ドリフトなし
sigma_gbm = 0.05  # ボラティリティ
T0 = 10.0  # 初期気温

for i in range(n_show):
    Z = rng.standard_normal(n_compare - 1)
    S = np.zeros(n_compare)
    S[0] = T0
    for j in range(n_compare - 1):
        S[j+1] = S[j] * np.exp((mu_gbm - 0.5*sigma_gbm**2) + sigma_gbm * Z[j])
    axes[1].plot(range(n_compare), S, alpha=0.6, linewidth=0.8)

axes[1].axhline(T0, color="red", linewidth=2, linestyle="--", label=f"Initial = {T0}°C")
axes[1].set_title("GBM (Stock Price Model) — Wrong for Temperature!")
axes[1].set_xlabel("Days")
axes[1].set_ylabel("Temperature (°C)")
axes[1].legend()
axes[1].set_ylim(-20, 40)

plt.tight_layout()
plt.show()

print("左 (OU): パスは季節成分 θ(t) の周りを変動し、戻ってくる")
print("右 (GBM): パスは発散しうる。季節性も平均回帰もない → 気温に不適切")

## 6. まとめと次のステップ

### このNotebookで学んだこと
1. 気温データは **季節成分** + **OU過程残差** に分解できる
2. **OU過程**は平均回帰する確率過程で、GBMと違い定常分布を持つ
3. パラメータ κ, σ は OLS回帰で推定でき、直感的な解釈がある
4. Euler-Maruyama法でシミュレーションパスを生成し、Fan Chartで可視化できる

### Step 3 (04_pricing.ipynb) への接続
- シミュレーションパスから HDD/CDD を計算 → 先物価格を算出
- **Market Price of Risk (λ)** を導入して、実測度 P からリスク中立測度 Q に変換
- Q測度下での SDE: $dT = [\kappa(\theta(t) - T) - \lambda\sigma] \, dt + \sigma \, dW^Q$